# Keyword Search Explorer

Interactive notebook to filter papers from [LeMat-Synth-Papers](https://huggingface.co/datasets/LeMaterial/LeMat-Synth-Papers).

**Pipeline:**
1. Load the three raw splits (`arxiv`, `omg24`, `chemrxiv`)
2. Pick a split
3. Explore & filter by **categories**
4. Filter by **keywords** (Option 1: direct string matching, Option 2: LLM-based)
5. Browse & export results

In [ ]:
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 120)

# Load the three raw splits from HuggingFace
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",
    split=None,
    token=True,
)

print("Available splits:")
for name, ds in dataset.items():
    print(f"  {name}: {len(ds)} papers")
print(f"\nColumns: {dataset[next(iter(dataset.keys()))].column_names}")

Available splits:
  arxiv: 62952 papers
  omg24: 16026 papers
  chemrxiv: 1962 papers
  thermocatalysis_keywords_only: 8378 papers
  thermocatalysis_keywords_and_LLM: 1333 papers

Columns: ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis']


---
## Step 1 — Choose split(s)

Pick one or more raw sources to combine.

In [70]:
from datasets import concatenate_datasets

# Pick one or more splits: "arxiv", "omg24", "chemrxiv"
SPLITS = ["arxiv", "omg24"]  # e.g. ["arxiv", "omg24"] to combine two sources


ds = concatenate_datasets([dataset[s] for s in SPLITS])
print(f"Selected splits {SPLITS}: {len(ds)} papers total")

Selected splits ['arxiv', 'omg24']: 78978 papers total


---
## Step 2 — Explore categories

The `categories` column is a string. Individual categories are separated by `,` or `;`.

- **arxiv**: codes like `cond-mat.mtrl-sci`, `physics.chem-ph`
- **chemrxiv**: names like `Solid State Chemistry`, `Surface`
- **omg24**: Semantic Scholar field-of-study labels

In [71]:
def parse_categories(cat_string):
    """Split a categories string by comma and semicolon into a list."""
    if cat_string is None:
        return []
    parts = []
    for part in cat_string.split(","):
        for sub_part in part.split(";"):
            cleaned = sub_part.strip()
            if cleaned:
                parts.append(cleaned)
    return parts


# Flatten all categories across the selected split(s)
all_cats = []
for cat_string in ds["categories"]:
    all_cats.extend(parse_categories(cat_string))

cat_counts = pd.Series(all_cats).value_counts()
print(f"{len(cat_counts)} unique categories across {len(ds)} papers\n")
print("Top 20 categories:")
print(cat_counts.head(20).to_string())

915 unique categories across 78978 papers

Top 20 categories:
Nanomaterials             12331
Superconductors           12148
Magnetic                  11937
Semiconductors            10685
Ceramics                   6661
Polymers                   2163
Metals                     1915
Other                      1347
Composites                 1256
Biomaterials                924
Chemicals                   640
Magnetics                   129
N/A                          86
Topological Insulators       82
Superconductor               58
Polymer                      58
Intermetallics               38
Liquid Crystals              31
Multiferroics                29
Organic Semiconductors       27


## Step 3 — Filter by categories

Keep only papers whose `categories` string contains at least one match.  
Uses substring matching (case-insensitive).  
Set `CATEGORY_FILTER = []` to skip and keep all papers.

In [62]:
# Categories to keep (substring match, case-insensitive)

# Examples:
#   arxiv:    ["cond-mat"]
#   chemrxiv: ["Solid State Chemistry", "Surface"]
CATEGORY_FILTER = ["Superconductor"]  # empty = keep all papers


def filter_by_category(ds, category_filter):
    """Keep papers whose categories string contains any of the filter terms."""
    if not category_filter:
        return ds
    cat_lower = [c.lower() for c in category_filter]

    def has_category(example):
        cats = example["categories"]
        if cats is None:
            return False
        cats_lower = cats.lower()
        return any(cf in cats_lower for cf in cat_lower)

    return ds.filter(has_category)


ds_cat = filter_by_category(ds, CATEGORY_FILTER)

if CATEGORY_FILTER:
    print(
        f"Category filter {CATEGORY_FILTER}: {len(ds_cat)} / {len(ds)} papers"
    )
else:
    print(f"No category filter applied: {len(ds_cat)} papers")

Filter:   0%|          | 0/62952 [00:00<?, ? examples/s]

Category filter ['Superconductor']: 12327 / 62952 papers


---
## Step 4 — Keyword filter

### Option 1: Direct keyword matching (default)

Papers matching **any** include keyword (and **none** of the exclude keywords) are kept.

In [65]:
# Column to search in
TEXT_COLUMN = "abstract"

# Include keywords — papers matching ANY of these are kept
INCLUDE_KEYWORDS = ["resistivity", "resistance", "critical temperature", "Tc"]

EXCLUDE_KEYWORDS = ["semiconductor"]

# How many papers to display
NUM_DISPLAY = 20

In [66]:
def keyword_filter(ds, text_column, include_kws, exclude_kws=None):
    """Filter HuggingFace Dataset by include/exclude keywords."""

    def matches_include(example):
        if example[text_column] is None:
            return False
        text_lower = example[text_column].lower()
        return any(kw.lower() in text_lower for kw in include_kws)

    filtered = ds.filter(matches_include)

    if exclude_kws:

        def matches_exclude(example):
            if example[text_column] is None:
                return False
            text_lower = example[text_column].lower()
            return any(ek.lower() in text_lower for ek in exclude_kws)

        filtered = filtered.filter(lambda x: not matches_exclude(x))

    return filtered


ds_filtered = keyword_filter(
    ds_cat, TEXT_COLUMN, INCLUDE_KEYWORDS, EXCLUDE_KEYWORDS
)
print(f"Keyword filter: {len(ds_filtered)} / {len(ds_cat)} papers")

Filter:   0%|          | 0/12327 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5363 [00:00<?, ? examples/s]

Keyword filter: 5286 / 12327 papers


### Option 2: LLM-based keyword filter (commented out)

Instead of exact string matching, use an LLM to decide whether a paper is relevant.  
Requires a running **vLLM** endpoint. Uncomment the cell below to use it.

**Prerequisites:**
- A vLLM server running locally (default: `http://localhost:8000/v1`)
- `pip install openai transformers`

In [67]:
# # --- Option 2: LLM-based filtering ---
# # Uncomment this entire cell to use LLM filtering instead of direct keywords.
# # Make sure to skip the Option 1 cells above.
#
# import openai
# from tqdm import tqdm
# from transformers import AutoTokenizer
#
# VLLM_ENDPOINT = "http://localhost:8000/v1"
# MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
# MAX_MODEL_LEN = 50000
#
# # Edit this prompt to match your use case
# LLM_PROMPT = """You are provided with a scientific materials paper.
# Read the paper carefully and determine if it is relevant to the topic
# of heterogeneous catalysis with temperature-dependent performance data.
#
# Answer with only yes or no.
# If you are not sure, answer with no.
#
# Paper: {paper_text}
# Question: Is this paper relevant?
# Answer:"""
#
# client = openai.OpenAI(base_url=VLLM_ENDPOINT, api_key="not-needed")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
#
# def ask_llm(text):
#     """Send paper text to LLM and return True if relevant."""
#     tokens = tokenizer.encode(text)
#     if len(tokens) > MAX_MODEL_LEN:
#         text = tokenizer.decode(tokens[:MAX_MODEL_LEN - 150])
#     message = LLM_PROMPT.format(paper_text=text)
#     try:
#         response = client.chat.completions.create(
#             model=MODEL_NAME,
#             messages=[{"role": "user", "content": message}],
#             temperature=0,
#             max_tokens=100,
#         )
#         answer = response.choices[0].message.content.strip().lower()
#         return "yes" in answer
#     except Exception as e:
#         print(f"LLM call failed: {e}")
#         return False
#
#
# # Run LLM filter on the category-filtered dataset (ds_cat)
# llm_results = []
# for paper in tqdm(ds_cat, desc="LLM filtering"):
#     text = paper.get("text_paper") or paper.get("abstract") or ""
#     llm_results.append(ask_llm(text))
#
# ds_cat_with_labels = ds_cat.add_column("llm_relevant", llm_results)
# ds_filtered = ds_cat_with_labels.filter(lambda x: x["llm_relevant"])
# print(f"LLM filter: {len(ds_filtered)} / {len(ds_cat)} papers")

---
## Step 5 — Results

### Per-keyword match counts

In [68]:
# Per-keyword counts on the filtered dataset
def count_keyword_matches(ds, text_column, keywords):
    counts = {}
    for kw in keywords:

        def has_keyword(example, keyword=kw):
            if example[text_column] is None:
                return False
            return keyword.lower() in example[text_column].lower()

        count = len(ds.filter(has_keyword))
        counts[kw] = count
    return counts


keyword_counts = count_keyword_matches(
    ds_filtered, TEXT_COLUMN, INCLUDE_KEYWORDS
)
print("Per-keyword match counts:")
for kw, count in sorted(keyword_counts.items(), key=lambda x: -x[1]):
    print(f"  {kw}: {count}")
print(f"\nTotal papers: {len(ds_filtered)}")

Filter:   0%|          | 0/5286 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5286 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5286 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5286 [00:00<?, ? examples/s]

Per-keyword match counts:
  Tc: 2900
  resistivity: 1910
  resistance: 1168
  critical temperature: 807

Total papers: 5286


### Browse results

In [69]:
DISPLAY_COLUMNS = ["id", "title", "categories", "abstract", "pdf_url"]

cols = [c for c in DISPLAY_COLUMNS if c in ds_filtered.column_names]
ds_filtered.to_pandas()[cols].head(NUM_DISPLAY)

,id,title,categories,abstract,pdf_url
0,0704.0529,Scanning Tunneling Spectroscopy in the Superconducting State and Vortex\n Cores of the beta-pyrochlore KOs2O6,Superconductors,We performed the first scanning tunneling spectroscopy measurements on the\npyrochlore superconductor KOs2O6 (Tc =...,http://arxiv.org/pdf/0704.0529v1
1,0704.1425,Measurements and analysis of the upper critical field $H_{c2}$ on an\n underdoped and overdoped $La_{2-x}Sr_xCuO_4$...,Superconductors,The upper critical field $H_{c2}$ is one of the many non conventional\nproperties of high-$T_c$ cuprates. It is po...,http://arxiv.org/pdf/0704.1425v2
2,0704.1528,Extremely strong-coupling superconductivity and anomalous lattice\n properties in the beta-pyrochlore oxide KOs2O6,Superconductors,Superconducting and normal-state properties of the beta-pyrochlore oxide\nKOs2O6 are studied by means of thermodyn...,http://arxiv.org/pdf/0704.1528v1
3,0704.3526,MgB2 single crystals substituted with Li and with Li-C: Structural and\n superconducting properties,Superconductors,The effect of Li substitution for Mg and of Li-C co-substitution on the\nsuperconducting properties and crystal st...,http://arxiv.org/pdf/0704.3526v1
4,0705.1239,Stray-fields-based magnetoresistance mechanism in Ni80Fe20-Nb-Ni80Fe20\n trilayers,Superconductors,We report on the transport and magnetic properties of hybrid trilayers and\nbilayers that consist of low spin-pola...,http://arxiv.org/pdf/0705.1239v1
5,0705.1401,Manifestations of fine features of the density of states in the\n transport properties of KOs2O6,Superconductors,"We performed high-pressure transport measurements on high-quality single\ncrystals of KOs2O6, a beta-pyrochlore su...",http://arxiv.org/pdf/0705.1401v1
6,0705.1438,Study and optimization of ion-irradiated High-Tc Josephson nanoJunctions\n by Monte Carlo simulations,Superconductors,High Tc Josephson nanoJunctions (HTc JnJ) made by ion irradiation have\nremarkable properties for technological ap...,http://arxiv.org/pdf/0705.1438v1
7,0705.1472,Uniform Mixing of Antiferromagnetism and High-Tc Superconductivity in\n Electron-doped Layers in Four-layered Ba2Ca...,Superconductors,We report Cu- and F-NMR studies on a four-layered high-temperature\nsuperconductor Ba2Ca3Cu4O8F2(0234F(2.0)) with ...,http://arxiv.org/pdf/0705.1472v2
8,0705.1638,Possible manifestation of spin fluctuations in the temperature behavior\n of resistivity in Sm_{1.85}Ce_{0.15}CuO_4...,Superconductors,A pronounced step-like (kink) behavior in the temperature dependence of\nresistivity $\rho (T)$ is observed in the...,http://arxiv.org/pdf/0705.1638v1
9,0705.2592,Incoherent non-Fermi liquid scattering in a Kondo lattice,Superconductors,One of the most notorious non-Fermi liquid properties of both archetypal\nheavy-fermion systems [1-4] and the high...,http://arxiv.org/pdf/0705.2592v2


### Export (optional)

In [ ]:
# df = ds_filtered.to_pandas()
# df.to_csv("filtered_papers.csv", index=False)
# df.to_pickle("filtered_papers.pkl")